# ParkCast SF — 311 Parking Complaint Features

311 cases are a real-time demand-pressure proxy that the meter transaction
feed can't capture: "blocked driveway", "double parked", "abandoned vehicle"
reports happen *because* parking is tight. They're available 24/7 — covering
the nights, weekends, and non-metered blocks where our current data goes dark.

For every SFMTA blockface, we compute the median daily complaint count for
each (hour-of-day, weekday) slot over the past 12 months.

**Output:** `data/block_311_features.csv` keyed on `(blockface_id, hour_of_day, weekday)`.

## Imports

In [ ]:
import os
import json
import time
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

from _http import retry_session

## Configuration

In [2]:
PROJECT_DIR = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(PROJECT_DIR, "data")
METERS = os.path.join(DATA_DIR, "meter_locations.csv")
CACHE = os.path.join(DATA_DIR, "sf_311_parking.csv")
OUT = os.path.join(DATA_DIR, "block_311_features.csv")

# DataSF Socrata endpoint. vw6y-z8j6 = 311 Cases dataset.
API_URL = "https://data.sfgov.org/resource/vw6y-z8j6.json"
PAGE = 50_000
LOOKBACK_DAYS = 365

# Real parking-related 311 categories in vw6y-z8j6 (checked against the
# last 12 months of counts). These are the only two that meaningfully
# signal parking demand pressure; Graffiti / Encampment / Cleaning etc.
# aren't driver-induced.
PARKING_SERVICES = [
    "Parking Enforcement",          # ~174k / yr — double-parked, expired, etc.
    "Blocked Street and Sidewalk",  # ~15k / yr — cars blocking driveways / crosswalks
]

## `fetch_311()`

Paginated Socrata pull. Filters server-side to the last 12 months and parking
keywords. Cached to `data/sf_311_parking.csv` so re-runs don't re-download.

In [ ]:
def fetch_311():
    if os.path.exists(CACHE):
        print(f"Loading cached 311 data from {CACHE}")
        return pd.read_csv(CACHE, parse_dates=["requested_datetime"])
    cutoff = (pd.Timestamp.utcnow() - pd.Timedelta(days=LOOKBACK_DAYS)).strftime("%Y-%m-%dT00:00:00")
    # SoQL doesn't support `OR` in $q the way we expect; use explicit IN(...)
    # on service_name instead. Much more reliable than keyword search.
    svc_list = ", ".join(f"'{s}'" for s in PARKING_SERVICES)
    where = (f"requested_datetime > '{cutoff}' "
             f"AND lat IS NOT NULL "
             f"AND service_name IN ({svc_list})")
    print(f"Querying DataSF 311 since {cutoff} ...")
    print(f"  Filter: service_name IN ({svc_list})")
    sess = retry_session()
    rows = []
    offset = 0
    while True:
        params = {
            "$select": "requested_datetime, service_name, lat, long",
            "$where": where,
            "$order": "requested_datetime",  # stable ordering for pagination
            "$limit": PAGE,
            "$offset": offset,
        }
        r = sess.get(API_URL, params=params, timeout=180)
        r.raise_for_status()
        batch = r.json()
        if not batch:
            break
        rows.extend(batch)
        print(f"  offset={offset:,}  fetched={len(batch):,}  total={len(rows):,}")
        if len(batch) < PAGE:
            break
        offset += PAGE
        time.sleep(0.5)
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError("No 311 rows returned — check API schema / filter.")
    df["requested_datetime"] = pd.to_datetime(df["requested_datetime"], errors="coerce")
    df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
    df["long"] = pd.to_numeric(df["long"], errors="coerce")
    df = df.dropna(subset=["requested_datetime", "lat", "long", "service_name"])
    df.to_csv(CACHE, index=False)
    print(f"  Cached {len(df):,} rows → {CACHE}")
    return df

## `filter_parking()`

Narrow the server-side superset to exact parking service categories. Using
both an exact-match set and a keyword fallback keeps the filter robust to
DataSF's occasional category renames.

In [4]:
def filter_parking(df):
    """Server-side filter already narrowed to parking categories; this
    just reports the distribution for sanity."""
    print(f"  Rows (already server-filtered): {len(df):,}")
    print("  service_name distribution:")
    print(df["service_name"].value_counts().head(5).to_string())
    return df

## `blockfaces_from_meters()`

SFMTA blockface centroids (same mapping the occupancy pipeline uses). We keyed
the output on `blockface_id` so `merge_citations`-style joins can reuse it
directly inside `preprocess_real_data.ipynb`.

In [5]:
def blockfaces_from_meters():
    locs = pd.read_csv(METERS, usecols=[
        "post_id", "blockface_id", "latitude", "longitude",
    ])
    locs = locs.dropna(subset=["blockface_id", "latitude", "longitude"])
    bf = (locs.groupby("blockface_id")
              .agg(lat=("latitude", "mean"), lon=("longitude", "mean"))
              .reset_index())
    print(f"  Blockfaces with coords: {len(bf):,}")
    return bf

## `aggregate_features()`

For each 311 case, find the nearest blockface (≤50m). Then aggregate:

1. Per (blockface, date, hour_of_day) — daily count.
2. Per (blockface, hour_of_day, weekday) — **median** across dates.

Median (not mean) keeps rare-spike neighborhoods from dominating the signal;
most blocks have zero complaints most hours, so median = 0 for typical slots
and non-zero where complaints are *routine*.

In [6]:
LAT0 = 37.77
M_PER_DEG_LAT = 111_000.0
M_PER_DEG_LON = 111_000.0 * np.cos(np.radians(LAT0))

def to_xy(lat, lon):
    y = (lat - LAT0) * M_PER_DEG_LAT
    x = (lon + 122.4194) * M_PER_DEG_LON
    return x, y

def aggregate_features(cases, blockfaces, match_m=75.0):
    bx, by = to_xy(blockfaces["lat"].values, blockfaces["lon"].values)
    tree = cKDTree(np.column_stack([bx, by]))

    cx, cy = to_xy(cases["lat"].values, cases["long"].values)
    dists, idx = tree.query(np.column_stack([cx, cy]), k=1)
    close = dists <= match_m
    cases = cases[close].copy()
    cases["blockface_id"] = blockfaces.iloc[idx[close]]["blockface_id"].values
    print(f"  Cases matched to blockface (≤{match_m:.0f}m): {len(cases):,}")

    cases["date"] = cases["requested_datetime"].dt.date
    cases["hour_of_day"] = cases["requested_datetime"].dt.hour
    cases["weekday"] = cases["requested_datetime"].dt.weekday

    daily = (cases.groupby(["blockface_id", "date", "hour_of_day", "weekday"])
                  .size().reset_index(name="n"))

    feats = (daily.groupby(["blockface_id", "hour_of_day", "weekday"])["n"]
                  .median()
                  .reset_index(name="complaints_311_median"))
    total = (daily.groupby("blockface_id")["n"].sum()
                  .reset_index(name="complaints_311_total"))
    feats = feats.merge(total, on="blockface_id", how="left")
    return feats

## Pipeline

In [7]:
print("=" * 60)
print("ParkCast SF — 311 Parking Complaint Features")
print("=" * 60)

raw = fetch_311()
print(f"Raw rows fetched: {len(raw):,}")

cases = filter_parking(raw)

bfs = blockfaces_from_meters()
feats = aggregate_features(cases, bfs)

feats.to_csv(OUT, index=False)
print(f"\nFeature rows (blockface × hour × weekday): {len(feats):,}")
print("  Summary:")
print(feats[["complaints_311_median", "complaints_311_total"]].describe().round(2).to_string())
print(f"\nSaved → {OUT}")
print("=" * 60)

ParkCast SF — 311 Parking Complaint Features
Querying DataSF 311 since 2025-04-19T00:00:00 ...
  Filter: service_name IN ('Parking Enforcement', 'Blocked Street and Sidewalk')


  offset=0  fetched=50,000  total=50,000


  offset=50,000  fetched=50,000  total=100,000


  offset=100,000  fetched=50,000  total=150,000


  offset=150,000  fetched=39,665  total=189,665


  Cached 189,665 rows → /Users/kayvan/Desktop/MSDS/ml-ops/Project/ml-ops-final-project-team-ParkCast-SF/data/sf_311_parking.csv
Raw rows fetched: 189,665
  Rows (already server-filtered): 189,665
  service_name distribution:
service_name
Parking Enforcement            174279
Blocked Street and Sidewalk     15386
  Blockfaces with coords: 3,091
  Cases matched to blockface (≤75m): 53,588

Feature rows (blockface × hour × weekday): 34,864
  Summary:
       complaints_311_median  complaints_311_total
count               34864.00              34864.00
mean                    1.06                 59.73
std                     0.34                123.91
min                     1.00                  1.00
25%                     1.00                 14.00
50%                     1.00                 30.00
75%                     1.00                 57.00
max                    16.50               1177.00

Saved → /Users/kayvan/Desktop/MSDS/ml-ops/Project/ml-ops-final-project-team-ParkCast-SF/